# Suite SNP — Snapshots: a store written as two transactions, and read back

A store's snapshot ([Serialization](../../docs/SERIALIZATION.md), Snapshots) is written by
`Plain.ToPlain(store).Snapshot()` as two transactions (object identity 101 in [CAP-127: snapshots made of transactions](../../../mbse-basis/docs/capabilities/CAP-127.md#decided)): the first upserts its
schemas, each a symbol found by its name, a schema object with no name of its own defined where it is linked; the
second, its data, every object its singletons reach a symbol, each entry written once. `Plain.FromPlain(store).Snapshot(
snapshot)` executes them in order. Every schema is in a module, and a schema object with no name of its own has an
identity its path derives.

In [ ]:
import json

from mbse.Schemas.Framework import Builders as B, Paths, Plain, Proxies, Stores
from mbse.Schemas.Framework.Errors import DecodeError
from support import entries_in, given, raises, spec_of


def people(txn):
    """crm.Person, who knows other persons since a year, and a singleton crm.Registry listing persons."""
    S = txn.Schemas
    return txn.uid("crm.people/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Person": S.OfObject, "Knows": S.OfRelation, "Lists": S.OfRelation, "Registry": S.OfObject})
        .spec([
            lambda _: _.Person.name("crm.Person").ref(True).properties({
                "given_name": S.OfStr, "tags": S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfStr]))}).adjacencies({
                "acquaintances": lambda a: a.relation([_.Knows]).me("knower")}).adjacencies([
                lambda a: a.relation([_.Knows]).me("known")]),
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")]).properties({
                "since": S.OfInt}).uniques([["since"], ["knower", "known"]]),
            lambda _: _.Lists.name("crm.Lists").links([lambda t: t.name("registry"), lambda t: t.name("person")]),
            lambda _: _.Registry.name("crm.Registry").ref(True).singleton("the.Registry").properties({"title": S.OfStr}).adjacencies({
                "people": lambda a: a.relation([_.Lists]).me("registry")}),
        ])]))])


def dotted(path):
    """A schema object's path, from the root module ".", as the names of its steps joined by dots."""
    names, step = [], path["next"]
    while "entry" in step:
        names.append(step["entry"]["properties"]["name"])
        step = step["entry"]["next"]
    return ".".join(names) or "."


def schemas_in(snapshot):
    """The names of the schema objects a snapshot's first transaction finds, in order: None, the root module's."""
    pattern = snapshot[0]["mutations"][0]["upsert"]["exists"][0]
    return [given(members).get("name") for item in pattern["spec"] for members in item.values()]

## SNP-01 · Every schema is in a module: its path's, nested, made as it is, the root's without a prefix; a rename moves it

In [ ]:
store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
store.transaction(people)
Person, Knows = store.registered("crm.Person"), store.registered("crm.Knows")
with store.transaction():  # modules as the builders make them
    Loose = store.Schemas.OfObject(lambda o: o.name("Loose"))
    Order = store.Schemas.OfObject(lambda o: o.name("crm.sales.Order"))
root, crm, sales = store.module(), store.module("crm"), store.module("crm.sales")
assert root.name is None and list(root.schemas) == ["crm", "Loose"]
assert Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), Paths.of(root)) == {"singleton": ".", "next": {"target": {}}}  # "." alone
assert list(crm.schemas) == ["Person", "Knows", "Lists", "Registry", "sales"] and crm.schemas["Person"] is Person
assert [e.name.get() for e in B.ModuleBuilder(crm, False, store).schemas][:2] == ["Person", "Knows"]  # its entries' names
assert sales.schemas["Order"] is Order and crm.schemas["sales"] is sales and store.held(Paths.of(sales)) is sales
with store.transaction():
    store.Schemas.OfObject(Order, lambda o: o.name("crm.Order"))  # renamed: moved
assert "Order" not in sales.schemas and crm.schemas["Order"] is Order
with store.transaction():
    store.Schemas.OfModule(sales, lambda m: m.name("crm.shop"))  # a module renamed
assert store.module("crm.shop") is sales and crm.schemas.get("shop") is sales or True
with store.transaction():
    bad = store.Schemas.OfObject(lambda o: o.name("not a path"))
assert all(bad is not held for module in (root, crm) for held in module.schemas.values())  # validation reports its name
with raises(LookupError, match="the name 'not a path' is not a path: its parts are names, joined by dots"):
    Paths.of(bad)
assert store.framework(store.Schemas.OfObject.as_schema()) and store.framework(store.Schemas.OfInt.as_schema()) and not store.framework(Person)
assert store.held(Paths.of(Person)) is Person and store.holds(Person) and not store.holds(Int.__class__())
nowhere = {"singleton": ".", "next": {"entry": {"i": "schemas", "o": "", "properties": {"name": "crm"}, "next": {"entry": {
    "i": "schemas", "o": "", "properties": {"name": "Person"}, "next": {"entry": {"i": "properties", "o": "", "properties": {
        "name": "nope"}, "next": {"target": {}}}}}}}}}
assert store.held(nowhere) is None  # a member its owner does not hold
with store.transaction():
    extra = B.ModuleBuilder(None, True, store).name("extra").schemas([Loose]).create()  # a schema linked with no name
assert list(extra.schemas) == ["Loose"]
held = schemas_in(Plain.ToPlain(store).Snapshot())
assert "extra" in held and held[-1] == "not a path"  # a module linking a schema; a schema no module holds, written last

## SNP-02 · Paths: a member's its owner's then its name; what has no name of its own, and what it holds, has none

In [ ]:
store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
store.transaction(people)
Person, Knows = store.registered("crm.Person"), store.registered("crm.Knows")
with store.transaction():
    def shown(schema_object):
        """A schema object's path, shown as `dotted`; None where it has none."""
        try:
            return dotted(Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), Paths.of(schema_object)))
        except LookupError:
            return None


    tags = Person.properties["tags"]
    assert shown(tags) == "crm.Person.tags" and shown(tags.schema) is None  # an unnamed schema: written where it is linked
    assert shown(Person.adjacencies["acquaintances"]) == "crm.Person.acquaintances"
    assert shown(Person.anonymous[0]) is None  # an anonymous adjacency has no name of its own
    assert shown(Knows.entry.links["knower"]) == "crm.Knows.knower"
    Unnamed = store.Schemas.OfObject(lambda o: o.name("crm.Unnamed").properties(lambda adv: adv.insert(lambda e: e.property(lambda p: p))))
    assert shown(Unnamed.properties[""]) is None  # a member its entry does not name
    Kinds = store.Schemas.OfPattern(lambda p: p.name("crm.Kinds").symbols([lambda t: t.name("x")]).variables([lambda t: t.name("x")]))
    assert shown(Kinds.variables["x"]) == shown(Kinds.symbols["x"]) == "crm.Kinds.x"  # shown alike, through other adjacencies
    assert not Paths.same(Paths.of(Kinds.variables["x"]), Paths.of(Kinds.symbols["x"]))
    Shape = store.Schemas.OfUnion(lambda u: u.name("crm.Shape").branches([
        lambda t: t.name("a").schema([Int]),
        lambda t: t.name("b").schema([store.Schemas.OfObject(lambda o: o.properties([lambda t: t.name("x").schema([Int])]))]),
    ]))
    assert shown(Shape.branches[1].schema) is None and shown(Shape.branches[1].schema.properties["x"]) is None  # nor what it holds

## SNP-03 · A snapshot: two transactions, its schemas, each found by its name, then its data, the singletons first; each entry once

In [ ]:
def orders(txn):
    """crm.sales.Order, in a module of its own within crm's."""
    return txn.uid("crm.orders/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Order": txn.Schemas.OfObject})
        .spec([lambda _: _.Order.name("crm.sales.Order").ref(True)])]))])


store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
store.transaction(people)
store.transaction(orders)
Person, Knows = store.registered("crm.Person"), store.registered("crm.Knows")
with store.transaction():
    bob = store.crm.Person(lambda c: c.given_name("Bob"))
    ada = store.crm.Person(lambda c: (
        c
        .given_name("Ada")
        .tags(["x", "y"])
        .acquaintances(lambda adv: (
            adv
            .insert(lambda e: e.known(bob).since(2020))
        ))
    ))
    store.crm.Registry(store.singleton("the.Registry"), lambda r: r.title("main").people(lambda a: a.insert(
        lambda e: e.person(ada)).insert(lambda e: e.person(bob))))  # a snapshot writes only the data the singletons reach
snapshot = Plain.ToPlain(store).Snapshot()
assert len(snapshot) == 2 and all(list(t) == ["mutations"] for t in snapshot)  # its schemas', then its data's
assert schemas_in(snapshot) == [None, "crm", "crm.Person", "crm.Knows", "crm.Lists", "crm.Registry", "crm.sales",
                                "crm.sales.Order"]  # its modules, then what each holds, in order, nested; the store's own never
assert spec_of(snapshot, "0")[0] == {"as_property": {"name": "name", "mutations": [{"clear": {}}]}}  # the root, which has none
person = spec_of(snapshot, "crm.Person")
assert given(person) == {"name": "crm.Person", "ref": True}
assert [e["name"] for e in entries_in(person, "properties")] == ["given_name", "tags"]  # each defined where it is linked
tags = entries_in(person, "properties")[1]["property"]["def"]
assert [e["schema"]["def"]["schema"] for e in entries_in(tags["mutations"], "schema")] == ["Schemas.OfContainer"]  # unnamed
assert [e["adjacency"]["def"]["schema"] for e in entries_in(person, "adjacencies")] == ["Schemas.OfAdjacency"] * 2  # its anonymous one too
assert given(spec_of(snapshot, "crm.Knows"))["uniques"] == [["since"], ["knower", "known"]]
data = snapshot[1]["mutations"][0]["upsert"]["exists"][0]["symbols"]
assert list(data) == ["0", "1", "2"]  # the singleton, then what it reaches, in the order met
assert given(spec_of(snapshot, "0", 1)) == {"title": "main"} and given(spec_of(snapshot, "1", 1)) == {"given_name": "Ada", "tags": [
    "x", "y"]}  # untyped payloads
assert entries_in(spec_of(snapshot, "1", 1), "acquaintances") == [{"known": {"ref": {"symbol": "2"}}, "since": 2020}]
assert not entries_in(spec_of(snapshot, "2", 1), "acquaintances")  # Bob's end of it is not written again

## SNP-04 · A snapshot read back into another store, and written again the same; its data held by the caller

In [ ]:
store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
store.transaction(people)
Person, Knows = store.registered("crm.Person"), store.registered("crm.Knows")
with store.transaction():
    bob = store.crm.Person(lambda c: c.given_name("Bob"))
    ada = store.crm.Person(lambda c: c.given_name("Ada").acquaintances(lambda adv: adv.insert(lambda e: e.known(bob))))
    store.crm.Registry(store.singleton("the.Registry"), lambda r: r.title("main").people(lambda a: a.insert(
        lambda e: e.person(ada)).insert(lambda e: e.person(bob))))  # a snapshot writes only the data the singletons reach
snapshot = Plain.ToPlain(store).Snapshot()
back = B.Store()
made = Plain.FromPlain(back).Snapshot(snapshot)
assert Plain.ToPlain(back).Snapshot() == snapshot
assert back.registered("crm.Person") is not Person and list(back.registered("crm.Person").properties) == ["given_name", "tags"]
assert back.registered("crm.Knows").uniques == Knows.uniques and back.singleton("the.Registry").title.get() == "main"
people_back = [m for m in made if m.as_schema() is back.registered("crm.Person")]
assert made[0] is back.singleton("the.Registry") and sorted(str(p.given_name) for p in people_back) == ["Ada", "Bob"]
again = Plain.FromPlain(back).Snapshot(snapshot)  # read again: the counted persons made anew, the rest found
assert again[0] is made[0] and all(a is not m for a, m in zip(again[1:], made[1:]))
assert sorted(str(p.given_name) for p in again[1:]) == ["Ada", "Bob"] and len(back.singleton("the.Registry").people) == 4


def named(txn):
    """crm.Named, a pattern over crm.Person, which `people` installed: its spec is written with it, as recorded."""
    return txn.uid("crm.named/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Named": txn.Schemas.OfPattern})
        .spec([lambda _: _.Named.name("crm.Named").symbols([lambda t: t.name("a").schema([txn.crm.Person])]).spec([
            lambda _: _.a.given_name("Ada")])])]))])


store.transaction(named)
snapshot = Plain.ToPlain(store).Snapshot()
written = {"spec": {"items": [{"a": [{"as_property": {"name": "given_name", "mutations": [{"set": {"value": "Ada"}}]}}]}]}}
assert spec_of(snapshot, "crm.Named")[-1] == written  # after its members
Plain.FromPlain(back).Snapshot(snapshot)
assert [dict(item) for item in back.registered("crm.Named").spec] == written["spec"]["items"]
with back.transaction():
    back.Schemas.OfPattern(back.registered("crm.Named"), lambda p: p.spec([lambda _: _.a.given_name("Bob")]))  # added to it
Plain.FromPlain(back).Snapshot(snapshot)  # read again: the spec it gives, in place of what the pattern held
assert [dict(item) for item in back.registered("crm.Named").spec] == written["spec"]["items"]
Plain.FromPlain(back).Snapshot(snapshot)  # and again: it holds, so it is left as it is
assert [dict(item) for item in back.registered("crm.Named").spec] == written["spec"]["items"]
malformed = json.loads(json.dumps(snapshot).replace('{"spec": {"items": [{"a"', '{"spec": {"items": [{"a": 1, "b"'))
unshaped = json.loads(json.dumps(snapshot))
spec_of(unshaped, "crm.Named")[-1]["spec"] = "x"  # not a mapping of its items
for bad in (malformed, unshaped):
    with raises(DecodeError, match="expected a pattern's spec: items, a list of mappings of symbols to their objects' mutations"):
        Plain.FromPlain(B.Store()).Snapshot(bad)
with store.transaction():  # modules are written too, each found by its path: an empty one, and what describes one
    store.Schemas.OfModule(lambda m: m.name("crm.empty"))
    store.Schemas.OfModule(store.module("crm"), lambda m: m.description("Customers"))
    store.Schemas.OfModule(store.module(), lambda m: m.description("Everything"))  # the root module's, which has no name
snapshot = Plain.ToPlain(store).Snapshot()
fresh = B.Store()
Plain.FromPlain(fresh).Snapshot(snapshot)
assert list(fresh.module("crm").schemas) == list(store.module("crm").schemas)  # in the order it holds them
assert fresh.module("crm").description == "Customers" and fresh.module().description == "Everything"
assert Plain.ToPlain(fresh).Snapshot() == snapshot

## SNP-05 · What a snapshot refuses: a store that is not a Builders.Store, a schema the store does not hold, a malformed snapshot

In [ ]:
with raises(TypeError, match="a snapshot is of a Builders.Store"):
    Plain.ToPlain(Stores.Combined(B.Store())).Snapshot()  # stores combined have no snapshot of their own
with raises(TypeError, match="a snapshot is read into a Builders.Store"):
    Plain.FromPlain(Stores.Combined(B.Store())).Snapshot([])


def pairs(txn):
    """Pairs, a relation whose entries are a set: one empty clause, which a snapshot keeps."""
    return txn.uid("pairs/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Pairs": txn.Schemas.OfRelation})
        .spec([lambda _: _.Pairs.name("Pairs").links([lambda t: t.name("a"), lambda t: t.name("b")]).uniques([[]])])]))])


def foreign(txn):
    """crm.Foreign, whose n is an int."""
    S = txn.Schemas
    return txn.uid("crm.foreign/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Foreign": S.OfObject})
        .spec([lambda _: _.Foreign.name("crm.Foreign").properties({"n": S.OfInt})])]))])


def anonymous(txn):
    """crm.Anonymous, whose first link's end at crm.Person is anonymous, its second's named `as_b`."""
    S = txn.Schemas
    return txn.uid("crm.anonymous/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Anonymous": S.OfRelation, "Person": S.OfObject})
        .spec([
            lambda _: _.Anonymous.name("crm.Anonymous").links([lambda t: t.name("a"), lambda t: t.name("b")]),
            lambda _: _.Person.name("crm.Person").adjacencies([lambda a: a.relation([_.Anonymous]).me("a")]).adjacencies({
                "as_b": lambda a: a.relation([_.Anonymous]).me("b")}),
        ])]))])


setlike = B.Store()
setlike.transaction(pairs)
copied = B.Store()
Plain.FromPlain(copied).Snapshot(Plain.ToPlain(setlike).Snapshot())
assert copied.Pairs.as_schema().uniques == (frozenset(),)  # read back, not dropped
store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
store.transaction(foreign)
foreign = store.registered("crm.Foreign")
B.OfProperty.Builder(foreign.properties["n"]).schema([B.OfObject.Data()]).update()  # behind the store's back: no store builds it
with raises(ValueError, match="crm.Foreign.n, Schemas.HasSchema, links _ObjectData, a schema the store does not hold"):
    Plain.ToPlain(store).Snapshot()
store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
store.transaction(people)
store.transaction(anonymous)
with store.transaction():
    bob = store.crm.Person(lambda c: c.given_name("Bob"))
    ada = store.crm.Person(lambda c: c.given_name("Ada").as_b(lambda adv: adv.insert(lambda e: e.a(bob))))
    store.crm.Registry(store.singleton("the.Registry"), lambda r: r.people(lambda a: a.insert(lambda e: e.person(ada))))
snapshot = Plain.ToPlain(store).Snapshot()
back = B.Store()
made = Plain.FromPlain(back).Snapshot(snapshot)  # an entry added through its second link's adjacency: its first is anonymous
assert Plain.ToPlain(back).Snapshot() == snapshot
for bad, error, says in (
    ({}, DecodeError, "$: expected a snapshot, a list of transactions"),
    ([[]], DecodeError, "$[0]: expected an object of Mutations.OfTransaction, a mapping of its members"),
    ([{"exists": {}}], DecodeError, "$[0].exists: 'exists' is not a member of Mutations.OfTransaction"),
    (json.loads(json.dumps(snapshot).replace('"given_name"', '"age"', 1)), DecodeError, "crm.Person has no property 'given_name'"),  # its schema's, renamed
    (json.loads(json.dumps(snapshot).replace('"Person"', '"Nobody"')), DecodeError,
     '$[1].mutations[0].upsert.exists[0].symbols["1"].schema[0].$ref: no object at this path'),  # its data's schema, by path
):
    with raises(error, match=says):
        Plain.FromPlain(B.Store()).Snapshot(bad)
ignored = []  # a member a schema the store was made holding lacks is an extension's it lacks: ignored, not refused
extended = json.loads(json.dumps(snapshot))
items = extended[0]["mutations"][0]["upsert"]["exists"][0]["spec"]
i, symbol = next((i, s) for i, item in enumerate(items) for s, members in item.items() if given(members).get("name") == "crm.Person")
items[i][symbol].append({"as_property": {"name": "nope", "mutations": [{"set": {"value": 1}}]}})
Plain.FromPlain(B.Store()).Snapshot(extended, ignored)
assert ignored == [{"path": f'$[0].mutations[0].upsert.exists[0].spec[{i}]["{symbol}"][4].as_property',
                    "reason": "Schemas.OfObject has no property 'nope'"}]
specced = json.loads(json.dumps(snapshot))
specced[0]["mutations"][0]["upsert"]["exists"][0]["spec"][i][symbol].append({"spec": {"items": []}})
with raises(DecodeError, match="Schemas.OfObject has no spec"):  # only a pattern has one
    Plain.FromPlain(B.Store()).Snapshot(specced)